# Team Challenge: Toolbox ML
## Construye tu propio paquete de herramientas para Data Science

---

### Introducción

En este Team Challenge vais a construir **desde cero un paquete de Python** llamado `toolbox_ml` que sirva como herramienta real de trabajo en proyectos de análisis exploratorio (EDA) y Machine Learning.

La diferencia con construir un script suelto es importante: en 2026 cualquier LLM puede generar una función básica de EDA en segundos. El verdadero aprendizaje está en **saber qué construir, estructurarlo correctamente, testearlo y documentarlo** de la misma manera en que lo haría un equipo de Data Science profesional. Al terminar este TC tendréis un paquete instalable, con tests que pasen y con documentación lista para enseñar en una entrevista de trabajo.

---

### Entregables

Se pide entregar:

1. **El paquete `toolbox_ml`** en el repositorio del grupo, con la estructura de directorios especificada más adelante, el código comentado y con docstrings en todas las funciones.

2. **Un directorio `tests/`** con tests unitarios escritos con `pytest` que cubran todas las funciones implementadas. Los tests deben pasar sin errores antes de la presentación.

3. **Un notebook de demostración** (`notebooks/demo.ipynb`) que muestre el uso de todas las funciones con un dataset real, con salidas visibles y comentarios explicativos.

4. **Un `README.md`** completo en la raíz del repositorio con descripción del paquete, instrucciones de instalación y ejemplos de uso.

5. **Una presentación de 10 minutos** en la que el grupo explica las decisiones de diseño tomadas, hace una demo en directo del código funcionando y ejecuta los tests en vivo.

---

### Plazos

Tendréis **2 sesiones de Team Challenge** para desarrollar el código y una sesión de presentaciones en el **Sprint 11**.

La entrega en el repositorio debe estar completa **el día antes de la presentación**.

- **Turno de mañana:** Defensa del TC → **11 de junio**
- **Turno de tarde:** Defensa del TC → **12 de junio**

El repositorio debe ser accesible para el profesor. Enviad el enlace antes de la presentación.

---

### Stack tecnológico

**Obligatorio:**
- **Python 3.10+**
- **pandas, numpy** — manipulación de datos
- **scipy** — tests estadísticos
- **matplotlib, seaborn** — visualización
- **scikit-learn** — modelos y métricas (necesario solo para la función bonus)
- **pytest** — tests unitarios
- **Git + GitHub** — control de versiones

Todas las dependencias deben estar recogidas en `requirements.txt`:

```
pandas>=2.2.0
numpy>=1.26.0
scipy>=1.12.0
matplotlib>=3.8.0
seaborn>=0.13.0
scikit-learn>=1.4.0
pytest>=8.0.0
```

---

### Estructura del paquete

El repositorio debe tener la siguiente estructura. Todos los ficheros marcados como obligatorios deben existir:

```
toolbox_ml/                   ← directorio raíz del paquete
├── __init__.py               ← obligatorio: hace el directorio importable
└── eda/
    ├── __init__.py           ← obligatorio
    └── core.py               ← aquí van todas las funciones
tests/
├── __init__.py
└── test_core.py              ← tests de todas las funciones
notebooks/
└── demo.ipynb                ← notebook de demostración
.gitignore
README.md
requirements.txt
setup.py
```

**El fichero `setup.py`** permite instalar el paquete localmente con `pip install -e .` y facilita importarlo desde cualquier lugar sin manipular el `PYTHONPATH`:

```python
# setup.py
from setuptools import setup, find_packages

setup(
    name='toolbox_ml',
    version='1.0.0',
    packages=find_packages(),
    install_requires=open('requirements.txt').read().splitlines()
)
```

Una vez instalado, cualquier función puede importarse así desde cualquier notebook o script:

```python
from toolbox_ml.eda.core import describe_df, tipifica_variables
```

---

### Docstrings y type hints

**Todas las funciones** deben incluir obligatoriamente:

**1. Type hints** en los argumentos y en el valor de retorno:

```python
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    ...
```

**2. Docstring** con descripción breve, sección de argumentos y sección de retorno:

```python
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen estadístico descriptivo de un DataFrame.

    Argumentos:
        df (pd.DataFrame): DataFrame a analizar.

    Retorna:
        pd.DataFrame: DataFrame con una fila por columna del input y las
        siguientes columnas: 'tipo', 'porcentaje_nulos', 'valores_unicos',
        'porcentaje_cardinalidad'.
        Retorna None si el input no es un DataFrame válido.
    """
    pass
```

**3. Comentarios en el cuerpo** de la función para explicar la lógica de cada bloque relevante.

---

## Funciones

Todas las funciones descritas a continuación son **obligatorias para todos los grupos** y deben implementarse en `toolbox_ml/eda/core.py`.

Para cada función se especifica la signatura completa, una descripción del comportamiento esperado, las comprobaciones de entrada que debe realizar y ejemplos orientativos. El diseño interno —cómo implementarlo— es decisión del grupo.

---

### Función: `describe_df`

**Signatura:**
```python
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
```

**Descripción:**
Recibe un DataFrame y devuelve otro DataFrame con una fila por cada columna del DataFrame original. El índice del resultado debe ser el nombre de cada columna. Las columnas del resultado deben ser:

| Columna | Contenido |
|---|---|
| `tipo` | Tipo de dato de la columna (`object`, `int64`, `float64`, `bool`, etc.) |
| `porcentaje_nulos` | Porcentaje de valores nulos o NaN sobre el total de filas |
| `valores_unicos` | Número de valores únicos distintos |
| `porcentaje_cardinalidad` | Porcentaje que representa el número de valores únicos sobre el total de filas |

**Comprobaciones de entrada:**
- Si `df` no es un `pd.DataFrame`, la función debe retornar `None` y mostrar por pantalla una descripción del error.

**Ejemplo de resultado con el dataset Titanic:**

```
              tipo  porcentaje_nulos  valores_unicos  porcentaje_cardinalidad
survived     int64               0.0               2                     0.22
pclass       int64               0.0               3                     0.34
name        object               0.0             891                   100.00
sex         object               0.0               2                     0.22
age        float64              19.87              88                     9.87
...
```

---

### Función: `tipifica_variables`

**Signatura:**
```python
def tipifica_variables(
    df: pd.DataFrame,
    umbral_categoria: int,
    umbral_continua: float
) -> pd.DataFrame:
```

**Descripción:**
Devuelve un DataFrame con dos columnas, `nombre_variable` y `tipo_sugerido`, con tantas filas como columnas tiene el DataFrame de entrada. La sugerencia del tipo se hace siguiendo esta lógica en cascada:

| Condición | Tipo sugerido |
|---|---|
| Cardinalidad igual a 2 | `"Binaria"` |
| Cardinalidad menor que `umbral_categoria` | `"Categórica"` |
| Cardinalidad mayor o igual a `umbral_categoria` **y** porcentaje de cardinalidad mayor o igual a `umbral_continua` | `"Numérica Continua"` |
| Cardinalidad mayor o igual a `umbral_categoria` **y** porcentaje de cardinalidad menor que `umbral_continua` | `"Numérica Discreta"` |

**Comprobaciones de entrada** (retornar `None` y mostrar el motivo si falla alguna):
- `df` debe ser un `pd.DataFrame`
- `umbral_categoria` debe ser un entero positivo
- `umbral_continua` debe ser un float entre 0 y 100

---

### Función: `get_features_num_regression`

**Signatura:**
```python
def get_features_num_regression(
    df: pd.DataFrame,
    target_col: str,
    umbral_corr: float,
    pvalue: float = None
) -> list:
```

**Descripción:**
Devuelve una lista con los nombres de las columnas numéricas del DataFrame cuya correlación de Pearson con `target_col` supere en valor absoluto el umbral `umbral_corr`.

Si `pvalue` es distinto de `None`, aplica un filtro adicional: solo se incluyen las columnas cuyo test de correlación de Pearson tenga un p-valor menor que `pvalue` (es decir, la correlación es estadísticamente significativa al nivel indicado).

**Pista estadística:**
`scipy.stats.pearsonr(x, y)` devuelve dos valores: el coeficiente de correlación y el p-valor. Un p-valor pequeño indica que es poco probable que esa correlación se deba al azar.

**Comprobaciones de entrada** (retornar `None` y mostrar el motivo si falla alguna):
- `df` debe ser un `pd.DataFrame`
- `target_col` debe existir en `df`
- `target_col` debe ser una columna numérica
- `umbral_corr` debe ser un float entre 0 y 1
- `pvalue`, si no es `None`, debe ser un float entre 0 y 1

---

### Función: `plot_features_num_regression`

**Signatura:**
```python
def plot_features_num_regression(
    df: pd.DataFrame,
    target_col: str = "",
    columns: list = [],
    umbral_corr: float = 0,
    pvalue: float = None
) -> list:
```

**Descripción:**
Pinta un pairplot del DataFrame considerando `target_col` y las columnas de `columns` que cumplan los criterios de correlación (misma lógica que `get_features_num_regression`). La función devuelve la lista de columnas que finalmente cumplen los criterios y se han representado.

Si `columns` está vacía, la función usa automáticamente todas las columnas numéricas del DataFrame como candidatas.

**Comportamiento cuando hay muchas columnas:** si la lista de columnas a representar supera 5 elementos, la función debe dividirla en grupos de máximo 5 columnas (incluyendo siempre `target_col` en cada grupo) y pintar un pairplot separado por cada grupo.

**Comprobaciones de entrada:** las mismas que `get_features_num_regression`. Si alguna falla, retornar `None` y mostrar el motivo.

---

### Función: `get_features_cat_regression`

**Signatura:**
```python
def get_features_cat_regression(
    df: pd.DataFrame,
    target_col: str,
    pvalue: float = 0.05
) -> list:
```

**Descripción:**
Devuelve una lista con los nombres de las columnas categóricas del DataFrame cuya relación estadística con `target_col` sea significativa al nivel `pvalue`.

La función debe **seleccionar automáticamente el test estadístico** más adecuado según la cardinalidad de cada variable categórica:

- Si la variable categórica tiene **exactamente 2 categorías**: usar el **test de Mann-Whitney U** (`scipy.stats.mannwhitneyu`), que compara las distribuciones de `target_col` entre los dos grupos.
- Si la variable categórica tiene **más de 2 categorías**: usar el **test ANOVA de un factor** (`scipy.stats.f_oneway`), que compara las medias de `target_col` entre todos los grupos.

En ambos casos, si el p-valor del test es menor que `pvalue`, la variable categórica tiene una relación estadísticamente significativa con el target y se incluye en el resultado.

**Comprobaciones de entrada** (retornar `None` y mostrar el motivo si falla alguna):
- `df` debe ser un `pd.DataFrame`
- `target_col` debe existir en `df`
- `target_col` debe ser numérica
- `pvalue` debe ser un float entre 0 y 1

---

### Función: `plot_features_cat_regression`

**Signatura:**
```python
def plot_features_cat_regression(
    df: pd.DataFrame,
    target_col: str = "",
    columns: list = [],
    pvalue: float = 0.05,
    with_individual_plot: bool = False
) -> list:
```

**Descripción:**
Para cada variable categórica de `columns` que supere el test estadístico correspondiente (misma lógica que `get_features_cat_regression`), pinta histogramas agrupados de `target_col` por cada valor de esa variable. La función devuelve la lista de columnas categóricas que han superado el test y se han representado.

Si `columns` está vacía, la función usa automáticamente todas las columnas categóricas del DataFrame como candidatas.

El argumento `with_individual_plot` controla el modo de visualización:
- `False` (por defecto): todas las variables se representan en una única figura con subplots.
- `True`: cada variable genera su propia figura independiente.

**Comprobaciones de entrada:** las mismas que `get_features_cat_regression`. Si alguna falla, retornar `None` y mostrar el motivo.

---

## Tests unitarios con pytest

**Todos los grupos** deben escribir tests para cada función en `tests/test_core.py`.

**Requisitos mínimos:**
- Al menos **3 tests por función**
- Los tests deben cubrir:
  - **Caso correcto:** input válido → output esperado
  - **Caso límite:** DataFrame vacío, columna con todos los valores nulos, etc.
  - **Caso de error:** input incorrecto → la función retorna `None`
- **Todos los tests deben pasar** antes de la presentación

**Cómo ejecutar los tests:**
```bash
# Desde la raíz del repositorio, con el entorno virtual activado
pytest tests/ -v
```

**Ejemplo de estructura de tests:**
```python
import pytest
import pandas as pd
from toolbox_ml.eda.core import describe_df


def test_describe_df_devuelve_dataframe():
    """Caso correcto: input válido → retorna DataFrame."""
    df = pd.DataFrame({'a': [1, 2, None], 'b': ['x', 'y', 'z']})
    resultado = describe_df(df)
    assert isinstance(resultado, pd.DataFrame)


def test_describe_df_columnas_correctas():
    """El DataFrame resultado tiene exactamente las columnas esperadas."""
    df = pd.DataFrame({'a': [1, 2, 3]})
    resultado = describe_df(df)
    assert set(resultado.columns) == {
        'tipo', 'porcentaje_nulos', 'valores_unicos', 'porcentaje_cardinalidad'
    }


def test_describe_df_porcentaje_nulos_correcto():
    """Calcula correctamente el porcentaje de nulos."""
    df = pd.DataFrame({'a': [1, None, None, None]})
    resultado = describe_df(df)
    assert resultado.loc['a', 'porcentaje_nulos'] == 75.0


def test_describe_df_retorna_none_con_input_invalido():
    """Caso de error: input no es DataFrame → retorna None."""
    assert describe_df("esto no es un dataframe") is None
    assert describe_df([1, 2, 3]) is None
```

**Nota:** al comparar floats en los tests, usar `pytest.approx()` para evitar fallos por precisión numérica:
```python
assert resultado.loc['a', 'porcentaje_nulos'] == pytest.approx(33.33, abs=0.01)
```

---

## README del repositorio

El `README.md` debe incluir al menos:

1. **Nombre y descripción** del paquete
2. **Instrucciones de instalación:**
   ```bash
   git clone https://github.com/vuestro-grupo/toolbox_ml.git
   cd toolbox_ml
   python -m venv venv
   source venv/bin/activate   # Mac/Linux
   pip install -r requirements.txt
   pip install -e .
   ```
3. **Ejemplo de uso de cada función** con código ejecutable
4. **Cómo ejecutar los tests:** `pytest tests/ -v`
5. **Descripción del equipo** y reparto de tareas

---

## Flujo de trabajo Git

Se aplica el mismo flujo del TC SQL: **feature branches + Pull Requests**, sin ramas `develop` ni `hotfix`.

1. Un integrante crea el repositorio y añade al equipo como colaboradores
2. **Proteger `main`**: Settings → Branches → Require PR antes de merge + 1 aprobación requerida
3. Cada integrante trabaja en su propia rama de feature:
   - `feature/describe-df`
   - `feature/tipifica-variables`
   - `feature/num-regression`
   - `feature/cat-regression`
   - `feature/tests`
4. Commits con **Conventional Commits**: `feat:`, `fix:`, `docs:`, `test:`
5. PR → code review → **Squash and merge**
6. Sincronizar `main` tras cada merge

**Al menos 1 PR mergeado por integrante** es requisito de entrega.

---

## Distribución del trabajo

**Para grupos de 3 personas:**

| Integrante | Responsabilidad |
|---|---|
| Scrum Master | Setup del repo, `setup.py`, `__init__.py`, integración final, notebook demo |
| Desarrollador 1 | `describe_df` + `tipifica_variables` + sus tests |
| Desarrollador 2 | `get/plot_features_num_regression` + `get/plot_features_cat_regression` + sus tests |

**Para grupos de 4 personas:**

| Integrante | Responsabilidad |
|---|---|
| Scrum Master | Setup del repo, `setup.py`, `__init__.py`, integración final, notebook demo |
| Desarrollador 1 | `describe_df` + `tipifica_variables` + sus tests |
| Desarrollador 2 | `get/plot_features_num_regression` + sus tests |
| Desarrollador 3 | `get/plot_features_cat_regression` + sus tests + función bonus |

**Consejo clave:** antes de empezar a codificar, acordad entre todos el formato de retorno de cada función y escribid primero los stubs (funciones vacías con `pass`) y los primeros tests. Así, mientras una persona implementa, otra puede testear. Este es el enfoque que se usa en equipos profesionales.

---

## Presentación

**Duración:** máximo **10 minutos** por grupo.

| Bloque | Tiempo | Contenido |
|---|---|---|
| Introducción | 1 min | Equipo, dataset elegido, reparto de tareas |
| Estructura del paquete | 1 min | Mostrar el repositorio y explicar las decisiones de diseño |
| Demo de las funciones | 5 min | Demo en directo del notebook con el dataset real |
| Tests en vivo | 2 min | Ejecutar `pytest tests/ -v` en directo |
| Retrospectiva | 1 min | Qué funcionó bien, qué cambiaríais |

**Lo más importante es la demo en directo.** No hay presentación de slides obligatoria; el código y el notebook son el protagonista.

Podéis usar cualquier dataset para la demostración. Algunos sugeridos:
- `seaborn.load_dataset('titanic')` — mezcla de variables numéricas y categóricas, con nulos
- `seaborn.load_dataset('diamonds')` — dataset de precios con variables ordinales
- `sklearn.datasets.fetch_california_housing()` — regresión con muchas variables numéricas
- Un dataset propio del TC SQL ShopNow cargado desde BigQuery

---

## Criterios de evaluación

| Criterio | Peso |
|---|---|
| Funciones implementadas correctamente | 40% |
| Tests unitarios presentes y pasando | 20% |
| Calidad del código (docstrings, type hints, comprobaciones de entrada, comentarios) | 20% |
| Presentación, demo en directo y respuesta a preguntas | 15% |
| README y estructura del paquete | 5% |

**Se penalizará:** código sin docstrings, tests que no pasan, ausencia de comprobaciones de entrada en las funciones y ausencia de historial de commits o PRs en el repositorio.

---

## Bonus (opcional)

Se valorará adicionalmente cualquiera de los siguientes puntos:

**1. Función `detect_outliers`**

Implementar en `toolbox_ml/eda/core.py` una función que detecte outliers en las columnas numéricas de un DataFrame usando al menos dos métodos: rango intercuartílico (IQR) y Z-score. La función debe devolver un diccionario con el número de outliers, su porcentaje y sus índices para cada columna analizada.

**2. CI con GitHub Actions**

Configurar un workflow `.github/workflows/tests.yml` que ejecute `pytest` automáticamente en cada push:

```yaml
name: Tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with: { python-version: '3.10' }
      - run: pip install -r requirements.txt && pip install -e .
      - run: pytest tests/ -v
```

**3. Funciones equivalentes para clasificación**

Implementar `get_features_num_classification` y `get_features_cat_classification` con la misma lógica que sus equivalentes de regresión, adaptando los tests estadísticos al caso en que el target es una variable categórica binaria o multiclase.

---

## Recursos

- [Documentación de scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) — tests estadísticos (pearsonr, mannwhitneyu, f_oneway)
- [Documentación de pytest](https://docs.pytest.org/) — cómo escribir y ejecutar tests
- [Guía de packaging en Python](https://packaging.python.org/en/latest/tutorials/packaging-projects/) — setup.py y estructura de paquetes
- [Conventional Commits](https://www.conventionalcommits.org/es/v1.0.0/) — formato de mensajes de commit
- La guía completa de entornos virtuales y Git está disponible en el fichero `guia_tc_sql.html` del TC anterior